#### Imports needed

In [2]:
import requests, json, io, os
from PIL import Image
from IPython.display import JSON
from datetime import datetime
from pyspark.sql import Row

StatementMeta(, e6827609-6b3b-46c1-adf3-ca4f3d46d164, 4, Finished, Available, Finished, False)

#### Credentials from Key Vault

In [3]:
vault_url = "https://cv-training-key.vault.azure.net/"
api_key = notebookutils.credentials.getSecret(vault_url, "det-obj-key")
endpoint = notebookutils.credentials.getSecret(vault_url, "det-obj-endpoint")

StatementMeta(, e6827609-6b3b-46c1-adf3-ca4f3d46d164, 5, Finished, Available, Finished, False)

#### Image analyzer

In [4]:
# Tagged Parameters cell
image_path = "test-object-detection/elsie_mas_2_people.jpg"  

StatementMeta(, e6827609-6b3b-46c1-adf3-ca4f3d46d164, 6, Finished, Available, Finished, False)

In [5]:
# --- Copy image from Lakehouse to /tmp/ ---
files_listing  = notebookutils.fs.ls("Files")
lakehouse_root = files_listing[0].path.split("/Files/")[0]
abs_image_path = f"{lakehouse_root}/Files/{image_path}"
notebookutils.fs.cp(abs_image_path, "file:/tmp/input.jpg")

StatementMeta(, e6827609-6b3b-46c1-adf3-ca4f3d46d164, 7, Finished, Available, Finished, False)

True

#### Call Azure AI Vision Object Detection AP

In [6]:
detect_url = f"{endpoint}/computervision/imageanalysis:analyze"

params  = {
    "features": "objects,tags,denseCaptions",   # objects returns bounding boxes
    "api-version": "2023-10-01"
}
headers = {
    "Ocp-Apim-Subscription-Key": api_key,
    "Content-Type": "application/octet-stream"
}

with open("/tmp/input.jpg", "rb") as img:
    response = requests.post(detect_url, params=params, headers=headers, data=img)

result = response.json()

StatementMeta(, e6827609-6b3b-46c1-adf3-ca4f3d46d164, 8, Finished, Available, Finished, False)

#### Explore data result to research

In [7]:
mi_dict = result
JSON(mi_dict)

StatementMeta(, e6827609-6b3b-46c1-adf3-ca4f3d46d164, 9, Finished, Available, Finished, False)

<IPython.core.display.JSON object>

#### Filter for animal/pet detections

In [8]:
PET_TAGS  = {"dog", "cat", "animal", "pet", "kitten", "puppy", "canine", "feline"}
THRESHOLD = 0.4

# Step 1 — Image-level tags
image_tags = {t["name"].lower() for t in result.get("tagsResult", {}).get("values", [])
                 if t["confidence"] >= THRESHOLD}
image_has_pet = bool(image_tags & PET_TAGS)

print(f"Image-level tags found:      {image_tags}")
print(f"Pet detected at image level: {image_has_pet}")

# Step 2 — Filter pet objects only
detections = []
for obj in result.get("objectsResult", {}).get("values", []):
    obj_tags = {t["name"].lower() for t in obj.get("tags", [])}
    obj_confidence = obj.get("tags", [{}])[0].get("confidence", 0)
    is_pet = bool(obj_tags & PET_TAGS) or (
        image_has_pet and not bool(obj_tags & {"person", "woman", "man", "girl", "boy", "child"})
    )
    if obj_confidence >= THRESHOLD and is_pet:
        detections.append(obj)

print(f"Pet objects found before deduplication: {len(detections)}")

# Step 3 — Deduplicate
def iou(box1, box2):
    x1 = max(box1["x"], box2["x"])
    y1 = max(box1["y"], box2["y"])
    x2 = min(box1["x"] + box1["w"], box2["x"] + box2["w"])
    y2 = min(box1["y"] + box1["h"], box2["y"] + box2["h"])
    intersection = max(0, x2 - x1) * max(0, y2 - y1)
    union = box1["w"]*box1["h"] + box2["w"]*box2["h"] - intersection
    return intersection / union if union > 0 else 0

def deduplicate(detections, iou_threshold=0.3):
    sorted_dets = sorted(detections, key=lambda o: o["tags"][0]["confidence"], reverse=True)
    kept = []
    for det in sorted_dets:
        if not any(iou(det["boundingBox"], k["boundingBox"]) > iou_threshold for k in kept):
            kept.append(det)
    return kept

detections = deduplicate(detections)
filename_base = os.path.splitext(os.path.basename(image_path))[0]
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
cropped_paths = []
rows = []

print(f"Pet objects after deduplication: {len(detections)}")

# Step 4 — Crop, save and build metrics rows together
if not detections:
    print("⚠️ No pet detected in image")
    rows.append(Row(
        image_name = f"{filename_base}_{timestamp}_no_detection.jpg",
        detected = False,
        animal_count = 0,
        object_name = "none",
        confidence = 0.0,
        bbox_x=0, bbox_y=0, bbox_w=0, bbox_h=0,
        timestamp = datetime.now()
    ))
else:
    for i, det in enumerate(detections):
        box = det["boundingBox"]
        tag = det["tags"][0]["name"]
        conf = det["tags"][0]["confidence"]
        cropped_name = f"{filename_base}_{timestamp}_{i}.jpg"

        with Image.open("/tmp/input.jpg") as img:
            cropped = img.crop((box["x"], box["y"], box["x"]+box["w"], box["y"]+box["h"]))
            local_out = f"/tmp/cropped_{i}.jpg"
            cropped.save(local_out)

        cropped_dest = f"{lakehouse_root}/Files/development/cropped/{cropped_name}"
        notebookutils.fs.cp(f"file:{local_out}", cropped_dest)
        cropped_paths.append(cropped_dest)

        rows.append(Row(
            image_name = cropped_name,
            detected = True,
            animal_count = len(detections),
            object_name = tag,
            confidence = float(conf),
            bbox_x = int(box["x"]),
            bbox_y = int(box["y"]),
            bbox_w = int(box["w"]),
            bbox_h = int(box["h"]),
            original_image_url = f"https://onelake.dfs.fabric.microsoft.com/lkh_pets.Lakehouse/Files/{image_path}",
            cropped_image_url = f"https://onelake.dfs.fabric.microsoft.com/lkh_pets.Lakehouse/Files/development/cropped/{cropped_name}",
            timestamp = datetime.now()
        ))

        print(f"✅ Crop {i+1}: '{tag}' ({conf:.2%}) → {cropped_name}")

    print(f"\n📦 Total crops saved: {len(cropped_paths)}")

StatementMeta(, e6827609-6b3b-46c1-adf3-ca4f3d46d164, 10, Finished, Available, Finished, False)

Image-level tags found:      {'indoor', 'dog breed', 'person', 'woman', 'clothing', 'human face', 'pet', 'dog'}
Pet detected at image level: True
Pet objects found before deduplication: 1
Pet objects after deduplication: 1
✅ Crop 1: 'dog' (88.00%) → elsie_mas_2_people_20260602_181602_0.jpg

📦 Total crops saved: 1


#### Save to object_detection_metrics table

In [9]:
#from pyspark.sql import Row

detection_df = spark.createDataFrame(rows)
detection_df.write.mode("append").saveAsTable("object_detection_metrics")

print(f"✅ Detection metrics saved: {len(rows)} row(s) for '{filename_base}'")

StatementMeta(, e6827609-6b3b-46c1-adf3-ca4f3d46d164, 11, Finished, Available, Finished, False)

✅ Detection metrics saved: 1 row(s) for 'elsie_mas_2_people'


#### List of cropped paths

In [10]:
# Pass cropped paths to the pipeline via notebook exit value
if cropped_paths:
    notebookutils.notebook.exit(json.dumps(cropped_paths))
else:
    notebookutils.notebook.exit(json.dumps([]))

StatementMeta(, e6827609-6b3b-46c1-adf3-ca4f3d46d164, 12, Finished, Available, Finished, False)

ExitValue: ["abfss://a96eab2a-8002-45f2-92b4-d2bf05c2540b@onelake.dfs.fabric.microsoft.com/db748fcd-0cc5-478c-a54c-f4938d542b0f/Files/development/cropped/elsie_mas_2_people_20260602_181602_0.jpg"]